# Per-Day Sampled Scatter Decoding Orchestrator — Patient Speech

Combines the per-day structure of `scat_classifier_sampled_nocv_filtered_speech_per_day.ipynb`
with the patient-speech filtering of `scat_classifier_sampled_nocv_filtered_patient_speech.ipynb`.

Words must pass **both** criteria:
1. **Quality-filtered** (`{patient}_word_df_filtered.csv`)
2. **Patient-speaking** (`{patient}_transcripts_annotated.csv` — LR-ASD marks segment as patient speech)

One SLURM job is submitted per *(patient, date, resample)* triplet.  
Outputs land in `{vad_root}/{patient}/standard_decoding_analysis/scat_xgboost_sampled_norm_patient_per_day/{date}/`.

**`REUSE_PATIENT_CONSENSUS = True`** (default): borrows hyperparameters from the already-finished
patient-level run (`scat_xgboost_sampled_norm_filtered_patient`) and skips the 3-resample hyper-search per day,
going straight to the 20-resample fixed-params stage.  
Set to `False` to run the full two-phase search independently for each day.

### 1. Imports

In [1]:
import json
import subprocess
from pathlib import Path

import dill as pickle
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

### 2. Configuration

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────────────────
PATIENTS    = ['YEY', 'YEZ', 'YFA', 'YFB', 'YFC', 'YFD', 'YFE', 'YFF', 'YFG',
               'YFI', 'YFK', 'YFL', 'YFM', 'YFN', 'YFP', 'YFQ', 'YFR', 'YFT', 'YFU', 'YFV']

VAD_ROOT    = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new')
LEGACY_ROOT = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_out')

BASE_RUN_NAME         = 'scat_xgboost_sampled_norm_patient_per_day'
PATIENT_CONSENSUS_RUN = 'scat_xgboost_sampled_norm_filtered_patient'   # source of borrowed params

CLUSTER_PREDS_OVERRIDE = {}
FRS_PATH_OVERRIDE      = {}

# ── Day-partition settings ────────────────────────────────────────────────────────────────
LOCAL_TZ           = 'America/Chicago'
ACTIVE_HOURS_START = 9
ACTIVE_HOURS_END   = 23
MIN_WORDS_PER_DAY  = 200

# ── Two-phase strategy ────────────────────────────────────────────────────────────────────
# True  → skip per-day hyperparameter search; borrow consensus from the patient-level run
# False → run full 3+17 pipeline independently per day
REUSE_PATIENT_CONSENSUS = False

N_RESAMPLES  = 20
FIRST_STAGE  = [0, 1, 2]
SECOND_STAGE = list(range(3, N_RESAMPLES))
SEED_STRIDE  = 42

PYTHON_BIN  = '/scratch/tahaismail424/miniforge3/envs/speech_247_env/bin/python3'
WORKER_PATH = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!'
                   '/standard_decoding_analysis/scat_classifier_worker.py')

PARTITION = 'Universal'
QOS       = 'big_batch_tier'
CPUS      = 8
MEM       = '40G'
TIME      = '48:00:00'

TEST_SIZE    = 0.2
SEARCH_ITERS = 40
CV_SPLITS    = 4
N_SHUFFLES   = 50
SCORING      = 'f1_macro'
PERM_METRIC  = 'f1_macro'

print('patients:', PATIENTS)
print('worker:', WORKER_PATH)
print(f'active hours: {ACTIVE_HOURS_START}:00 – {ACTIVE_HOURS_END}:00 {LOCAL_TZ}')
print(f'reuse patient consensus: {REUSE_PATIENT_CONSENSUS}')

patients: ['YEY', 'YEZ', 'YFA', 'YFB', 'YFC', 'YFD', 'YFE', 'YFF', 'YFG', 'YFI', 'YFK', 'YFL', 'YFM', 'YFN', 'YFP', 'YFQ', 'YFR', 'YFT', 'YFU', 'YFV']
worker: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_decoding_analysis/scat_classifier_worker.py
active hours: 9:00 – 23:00 America/Chicago
reuse patient consensus: False


### 3. Compute & Save Per-Day Patient-Speech Word Indices Per Patient

In [3]:
def compute_day_indices(patient: str) -> 'dict[str, Path]':
    """
    Load quality-filtered word df, intersect with patient-speaking segments,
    group by local calendar date within active hours, and save one .npy index
    file per day.  Returns {date_str: path}.
    """
    patient_root  = VAD_ROOT / patient
    filtered_csv  = patient_root / f'{patient}_word_df_filtered.csv'
    annotated_csv = patient_root / f'{patient}_transcripts_annotated.csv'

    if not filtered_csv.exists():
        print(f'  [{patient}] missing {filtered_csv.name}')
        return {}
    if not annotated_csv.exists():
        print(f'  [{patient}] missing {annotated_csv.name} — run add_video_diarization_df.ipynb first')
        return {}

    df           = pd.read_csv(filtered_csv, usecols=['original_word_idx', 'utc_word_start'])
    annotated_df = pd.read_csv(annotated_csv, index_col=0)

    patient_speaking = annotated_df['patient_speaking'].astype(bool)
    df['patient_speaking'] = patient_speaking.reindex(df['original_word_idx']).values
    df = df[df['patient_speaking'] == True].copy()

    if df.empty:
        print(f'  [{patient}] no patient-speaking words after quality filter')
        return {}

    df['utc_word_start'] = pd.to_datetime(df['utc_word_start'], utc=True)
    df['local_dt'] = df['utc_word_start'].dt.tz_convert(LOCAL_TZ)
    df['hour']     = df['local_dt'].dt.hour
    df['date_str'] = df['local_dt'].dt.strftime('%Y-%m-%d')

    active = df[(df['hour'] >= ACTIVE_HOURS_START) & (df['hour'] < ACTIVE_HOURS_END)]

    day_paths = {}
    for date_str, grp in active.groupby('date_str'):
        if len(grp) < MIN_WORDS_PER_DAY:
            print(f'  [{patient}] {date_str}: only {len(grp)} patient-speech words — skipping')
            continue
        idx_dir  = patient_root / 'standard_decoding_analysis' / BASE_RUN_NAME / date_str
        idx_dir.mkdir(parents=True, exist_ok=True)
        idx_path = idx_dir / f'{patient}_{date_str}_word_idx.npy'
        word_idx = grp['original_word_idx'].values.astype(np.int64)
        np.save(idx_path, word_idx)
        day_paths[date_str] = idx_path
        print(f'  [{patient}] {date_str}: {len(word_idx):,} patient-speech words → {idx_path.name}')

    return day_paths


print(f'Computing per-day patient-speech word indices for {len(PATIENTS)} patients...\n')
patient_day_idx_paths = {}
for patient in PATIENTS:
    day_paths = compute_day_indices(patient)
    if day_paths:
        patient_day_idx_paths[patient] = day_paths

total_pairs = sum(len(v) for v in patient_day_idx_paths.values())
print(f'\nTotal (patient, day) pairs ready: {total_pairs}')

Computing per-day patient-speech word indices for 20 patients...

  [YEY] missing YEY_transcripts_annotated.csv — run add_video_diarization_df.ipynb first
  [YEZ] missing YEZ_transcripts_annotated.csv — run add_video_diarization_df.ipynb first
  [YFA] missing YFA_transcripts_annotated.csv — run add_video_diarization_df.ipynb first
  [YFB] missing YFB_transcripts_annotated.csv — run add_video_diarization_df.ipynb first
  [YFC] 2024-07-20: 693 patient-speech words → YFC_2024-07-20_word_idx.npy
  [YFC] 2024-07-21: 1,963 patient-speech words → YFC_2024-07-21_word_idx.npy
  [YFC] 2024-07-22: 2,265 patient-speech words → YFC_2024-07-22_word_idx.npy
  [YFC] 2024-07-23: 2,427 patient-speech words → YFC_2024-07-23_word_idx.npy
  [YFC] 2024-07-24: 3,959 patient-speech words → YFC_2024-07-24_word_idx.npy
  [YFC] 2024-07-25: 4,340 patient-speech words → YFC_2024-07-25_word_idx.npy
  [YFC] 2024-07-26: 1,641 patient-speech words → YFC_2024-07-26_word_idx.npy
  [YFD] 2024-07-30: 209 patient-speech wo

### 4. Path Resolution & Consensus-Params Helpers

In [4]:
def first_existing(paths):
    for p in paths:
        if p is not None and Path(p).exists():
            return Path(p)
    return None


def resolve_patient_neural_inputs(patient: str) -> dict:
    root = VAD_ROOT / patient
    cluster_override = CLUSTER_PREDS_OVERRIDE.get(patient)
    frs_override     = FRS_PATH_OVERRIDE.get(patient)

    new_clusters = first_existing([root / 'embeddings' / 'semantic_cluster_predictions.npy'])
    new_frs = first_existing([
        root / 'neural_embeddings' / 'word_frs.npy',
        root / 'all_convo_recording' / 'word_avg_frs.npy',
    ])
    legacy_frs = first_existing([LEGACY_ROOT / patient / 'all_convo_recording' / 'word_avg_frs.npy'])

    if cluster_override is not None or frs_override is not None:
        cluster_preds_path = first_existing([cluster_override, new_clusters])
        frs_path           = first_existing([frs_override, new_frs, legacy_frs])
    elif new_clusters is not None and new_frs is not None:
        cluster_preds_path = new_clusters
        frs_path           = new_frs
    else:
        cluster_preds_path = None
        frs_path           = None

    return {'cluster_preds_path': cluster_preds_path, 'frs_path': frs_path}


def patient_consensus_path(patient: str) -> 'Path | None':
    """Return existing consensus params from the base patient-level run, if present."""
    p = VAD_ROOT / patient / 'standard_decoding_analysis' / PATIENT_CONSENSUS_RUN / 'consensus_best_params.json'
    return p if p.exists() else None


def choose_consensus_params(param_options: list) -> dict:
    params = dict(param_options[0])
    params['max_depth']        = int(np.min([p['max_depth'] for p in param_options]))
    params['min_child_weight'] = int(np.max([p['min_child_weight'] for p in param_options]))
    params['gamma']            = float(np.max([p['gamma'] for p in param_options]))
    params['reg_lambda']       = float(np.max([p['reg_lambda'] for p in param_options]))
    params['reg_alpha']        = float(np.max([p['reg_alpha'] for p in param_options]))
    params['subsample']        = float(np.median([p['subsample'] for p in param_options]))
    params['colsample_bytree'] = float(np.median([p['colsample_bytree'] for p in param_options]))
    params['learning_rate']    = float(np.median([p['learning_rate'] for p in param_options]))
    params['n_estimators']     = int(np.median([p['n_estimators'] for p in param_options]))
    params['tree_method'] = 'hist'
    params['device'] = 'cpu'
    params.pop('predictor', None)
    params.pop('use_label_encoder', None)
    return params

### 5. Build Status Table

In [5]:
def resample_paths(out_root: Path, r: int) -> dict:
    return {
        'pkl':              out_root / f'scat_sampled_resample_{r}_.pkl',
        'summary_json':     out_root / f'summary_resample_{r}.json',
        'best_params_json': out_root / f'best_params_resample_{r}.json',
        'success':          out_root / f'resample_{r}_SUCCESS',
        'error':            out_root / f'resample_{r}_error.txt',
    }


def build_status_df() -> pd.DataFrame:
    rows = []
    for patient in PATIENTS:
        neural = resolve_patient_neural_inputs(patient)
        day_idx_map = patient_day_idx_paths.get(patient, {})
        borrowed_consensus = patient_consensus_path(patient) if REUSE_PATIENT_CONSENSUS else None

        for date_str, word_idx_path in day_idx_map.items():
            out_root = VAD_ROOT / patient / 'standard_decoding_analysis' / BASE_RUN_NAME / date_str
            out_root.mkdir(parents=True, exist_ok=True)
            consensus_json = out_root / 'consensus_best_params.json'
            has_inputs = (neural['cluster_preds_path'] is not None
                         and neural['frs_path'] is not None)

            # When reusing patient consensus, copy it into the day output dir if not there yet
            if REUSE_PATIENT_CONSENSUS and borrowed_consensus is not None and not consensus_json.exists():
                consensus_json.write_text(borrowed_consensus.read_text())

            consensus_ready = consensus_json.exists()

            if REUSE_PATIENT_CONSENSUS:
                effective_first_stage  = []
                effective_second_stage = list(range(N_RESAMPLES))
            else:
                effective_first_stage  = FIRST_STAGE
                effective_second_stage = SECOND_STAGE

            first_three_done = all(
                (out_root / f'resample_{r}_SUCCESS').exists()
                for r in effective_first_stage
            ) if effective_first_stage else True

            for r in range(N_RESAMPLES):
                paths = resample_paths(out_root, r)
                rows.append({
                    'patient':             patient,
                    'date':                date_str,
                    'resample_idx':        r,
                    'stage':               'hyperparam' if r in effective_first_stage else 'fixed',
                    'seed':                r * SEED_STRIDE,
                    'cluster_preds_path':  neural['cluster_preds_path'],
                    'frs_path':            neural['frs_path'],
                    'word_idx_path':       word_idx_path,
                    'has_inputs':          has_inputs,
                    'out_root':            out_root,
                    'consensus_json':      consensus_json,
                    'consensus_ready':     consensus_ready,
                    'first_three_done':    first_three_done,
                    'pkl_path':            paths['pkl'],
                    'summary_json':        paths['summary_json'],
                    'best_params_json':    paths['best_params_json'],
                    'success_path':        paths['success'],
                    'error_path':          paths['error'],
                    'has_pkl':             paths['pkl'].exists(),
                    'has_success':         paths['success'].exists(),
                    'has_error':           paths['error'].exists(),
                    'ready_phase1':        (has_inputs and r in effective_first_stage
                                          and not paths['success'].exists()),
                    'ready_phase2':        (has_inputs and r in effective_second_stage
                                          and first_three_done and consensus_ready
                                          and not paths['success'].exists()),
                })
    return pd.DataFrame(rows).sort_values(['patient', 'date', 'resample_idx']).reset_index(drop=True)


status_df = build_status_df()
print(f'total rows (patient × day × resample): {len(status_df)}')
print(f'has inputs:      {int(status_df["has_inputs"].sum())}')
print(f'consensus ready: {int(status_df["consensus_ready"].sum())}')
print(f'completed:       {int(status_df["has_success"].sum())}')
print(f'errors:          {int(status_df["has_error"].sum())}')
print(f'ready_phase1:    {int(status_df["ready_phase1"].sum())}')
print(f'ready_phase2:    {int(status_df["ready_phase2"].sum())}')
status_df[['patient','date','resample_idx','stage','has_inputs','consensus_ready',
           'has_success','has_error','ready_phase1','ready_phase2']].head(40)

total rows (patient × day × resample): 1220
has inputs:      1220
consensus ready: 0
completed:       84
errors:          99
ready_phase1:    99
ready_phase2:    0


,patient,date,resample_idx,stage,has_inputs,consensus_ready,has_success,has_error,ready_phase1,ready_phase2
0,YFC,2024-07-20,0,hyperparam,True,False,False,True,True,False
1,YFC,2024-07-20,1,hyperparam,True,False,False,True,True,False
2,YFC,2024-07-20,2,hyperparam,True,False,False,True,True,False
3,YFC,2024-07-20,3,fixed,True,False,False,False,False,False
4,YFC,2024-07-20,4,fixed,True,False,False,False,False,False
5,YFC,2024-07-20,5,fixed,True,False,False,False,False,False
6,YFC,2024-07-20,6,fixed,True,False,False,False,False,False
7,YFC,2024-07-20,7,fixed,True,False,False,False,False,False
8,YFC,2024-07-20,8,fixed,True,False,False,False,False,False
9,YFC,2024-07-20,9,fixed,True,False,False,False,False,False


### 6. Patient × Day Input Summary

In [6]:
pair_df = (
    status_df.drop_duplicates(subset=['patient', 'date'])
    [['patient', 'date', 'has_inputs', 'consensus_ready', 'cluster_preds_path', 'frs_path', 'word_idx_path']]
    .sort_values(['patient', 'date'])
    .reset_index(drop=True)
)
print(f'{len(pair_df)} (patient, day) pairs')
display(pair_df)

61 (patient, day) pairs


,patient,date,has_inputs,consensus_ready,cluster_preds_path,frs_path,word_idx_path
0,YFC,2024-07-20,True,False,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
1,YFC,2024-07-21,True,False,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
2,YFC,2024-07-22,True,False,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
3,YFC,2024-07-23,True,False,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
4,YFC,2024-07-24,True,False,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
5,YFC,2024-07-25,True,False,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
6,YFC,2024-07-26,True,False,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
7,YFD,2024-07-30,True,False,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
8,YFD,2024-08-01,True,False,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
9,YFD,2024-08-02,True,False,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...


### 7a. Submit Phase-1 Jobs  
*(Only runs if `REUSE_PATIENT_CONSENSUS = False`; otherwise `ready_phase1` is always empty.)*

In [7]:
DRY_RUN = False

submitted = []
failed    = []

for _, row in status_df[status_df['ready_phase1']].iterrows():
    patient  = row['patient']
    date_str = row['date']
    r        = int(row['resample_idx'])
    logs_dir    = row['out_root'] / 'slurm_logs'
    scripts_dir = row['out_root'] / 'slurm_scripts'
    logs_dir.mkdir(parents=True, exist_ok=True)
    scripts_dir.mkdir(parents=True, exist_ok=True)

    word_idx_arg = f'--word-idx-path "{row["word_idx_path"]}"' if row['word_idx_path'] else ''
    sbatch_text = f"""#!/bin/bash
#SBATCH --job-name=scat_pspd_{patient}_{date_str}_{r}
#SBATCH --partition={PARTITION}
#SBATCH --qos={QOS}
#SBATCH --cpus-per-task={CPUS}
#SBATCH --mem={MEM}
#SBATCH --time={TIME}
#SBATCH --output={logs_dir}/{patient}_{date_str}_r{r}_%j.out
#SBATCH --error={logs_dir}/{patient}_{date_str}_r{r}_%j.err

set -eo pipefail
echo "HOSTNAME: $(hostname)"
echo "START: $(date)"
echo "PATIENT: {patient}"
echo "DATE: {date_str}"
echo "RESAMPLE: {r}"

{PYTHON_BIN} {WORKER_PATH} \\
  --patient {patient} \\
  --resample-idx {r} \\
  --cluster-preds-path "{row['cluster_preds_path']}" \\
  --frs-path "{row['frs_path']}" \\
  --out-dir "{row['out_root']}" \\
  {word_idx_arg} \\
  --test-size {TEST_SIZE} \\
  --n-iter {SEARCH_ITERS} \\
  --cv-splits {CV_SPLITS} \\
  --n-shuffles {N_SHUFFLES} \\
  --seed-stride {SEED_STRIDE} \\
  --n-jobs {CPUS * 2} \\
  --scoring {SCORING} \\
  --perm-metric {PERM_METRIC}

echo "END: $(date)"
"""
    sbatch_path = scripts_dir / f'{patient}_{date_str}_resample_{r}.sbatch'
    sbatch_path.write_text(sbatch_text)

    if DRY_RUN:
        submitted.append((patient, date_str, r, 'dry-run'))
        print(f'dry-run: {patient} {date_str} r={r}')
    else:
        try:
            res = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
            submitted.append((patient, date_str, r, res.stdout.strip()))
            print(f'submitted: {patient} {date_str} r={r} -> {res.stdout.strip()}')
        except subprocess.CalledProcessError as exc:
            failed.append((patient, date_str, r, exc.stderr.strip()))
            print(f'FAILED: {patient} {date_str} r={r}\n{exc.stderr}')

print(f'phase-1 submitted={len(submitted)}, failed={len(failed)}')

submitted: YFC 2024-07-20 r=0 -> Submitted batch job 295621
submitted: YFC 2024-07-20 r=1 -> Submitted batch job 295622
submitted: YFC 2024-07-20 r=2 -> Submitted batch job 295623
submitted: YFD 2024-07-30 r=0 -> Submitted batch job 295624
submitted: YFD 2024-07-30 r=1 -> Submitted batch job 295625
submitted: YFD 2024-07-30 r=2 -> Submitted batch job 295626
submitted: YFE 2024-08-13 r=0 -> Submitted batch job 295627
submitted: YFE 2024-08-13 r=1 -> Submitted batch job 295628
submitted: YFE 2024-08-13 r=2 -> Submitted batch job 295629
submitted: YFE 2024-08-14 r=0 -> Submitted batch job 295630
submitted: YFE 2024-08-14 r=1 -> Submitted batch job 295631
submitted: YFE 2024-08-14 r=2 -> Submitted batch job 295632
submitted: YFE 2024-08-15 r=0 -> Submitted batch job 295633
submitted: YFE 2024-08-15 r=1 -> Submitted batch job 295634
submitted: YFE 2024-08-15 r=2 -> Submitted batch job 295635
submitted: YFF 2024-08-21 r=0 -> Submitted batch job 295636
submitted: YFF 2024-08-21 r=1 -> Submitt

### 7b. Build Per-Day Consensus Params  
*(Only needed if `REUSE_PATIENT_CONSENSUS = False`; otherwise consensus was copied in the status step.)*

In [8]:
if not REUSE_PATIENT_CONSENSUS:
    built   = []
    blocked = []
    for (patient, date_str), grp in status_df.groupby(['patient', 'date']):
        out_root       = grp.iloc[0]['out_root']
        consensus_json = grp.iloc[0]['consensus_json']
        if consensus_json.exists():
            print(f'consensus already exists: {patient} {date_str}')
            continue
        first_three = grp[grp['resample_idx'].isin(FIRST_STAGE)]
        if not bool(first_three['has_success'].all()):
            blocked.append((patient, date_str))
            continue
        param_options = []
        for r in FIRST_STAGE:
            path = out_root / f'best_params_resample_{r}.json'
            param_options.append(json.loads(path.read_text()))
        consensus = choose_consensus_params(param_options)
        consensus_json.write_text(json.dumps(consensus, indent=2))
        built.append((patient, date_str))
        print(f'built consensus: {patient} {date_str}')
    print('built:', len(built), '  blocked:', len(blocked))
else:
    print('REUSE_PATIENT_CONSENSUS=True — consensus already copied during status build.')

built consensus: YFC 2024-07-21
built consensus: YFC 2024-07-22
built consensus: YFC 2024-07-23
built consensus: YFC 2024-07-24
built consensus: YFC 2024-07-25
built consensus: YFC 2024-07-26
built consensus: YFD 2024-08-01
built consensus: YFD 2024-08-02
built consensus: YFD 2024-08-03
built consensus: YFE 2024-08-16
built consensus: YFF 2024-08-23
built consensus: YFF 2024-08-24
built consensus: YFF 2024-08-26
built consensus: YFF 2024-08-27
built consensus: YFF 2024-08-28
built consensus: YFI 2024-10-16
built consensus: YFK 2025-02-13
built consensus: YFM 2025-03-13
built consensus: YFP 2025-05-06
built consensus: YFP 2025-05-07
built consensus: YFQ 2025-06-14
built consensus: YFQ 2025-06-18
built consensus: YFT 2025-07-29
built consensus: YFT 2025-07-30
built consensus: YFT 2025-07-31
built consensus: YFT 2025-08-02
built consensus: YFU 2025-12-14
built consensus: YFV 2026-02-20
built: 28   blocked: 33


### 7c. Submit Phase-2 (Fixed-Params) Jobs

In [9]:
DRY_RUN = False

# Refresh status after any consensus files were written
status_df = build_status_df()

submitted = []
failed    = []

for _, row in status_df[status_df['ready_phase2']].iterrows():
    patient  = row['patient']
    date_str = row['date']
    r        = int(row['resample_idx'])
    logs_dir    = row['out_root'] / 'slurm_logs'
    scripts_dir = row['out_root'] / 'slurm_scripts'
    logs_dir.mkdir(parents=True, exist_ok=True)
    scripts_dir.mkdir(parents=True, exist_ok=True)

    word_idx_arg = f'--word-idx-path "{row["word_idx_path"]}"' if row['word_idx_path'] else ''
    sbatch_text = f"""#!/bin/bash
#SBATCH --job-name=scat_pspd_{patient}_{date_str}_{r}
#SBATCH --partition={PARTITION}
#SBATCH --qos={QOS}
#SBATCH --cpus-per-task={CPUS}
#SBATCH --mem={MEM}
#SBATCH --time={TIME}
#SBATCH --output={logs_dir}/{patient}_{date_str}_r{r}_%j.out
#SBATCH --error={logs_dir}/{patient}_{date_str}_r{r}_%j.err

set -eo pipefail
echo "HOSTNAME: $(hostname)"
echo "START: $(date)"
echo "PATIENT: {patient}"
echo "DATE: {date_str}"
echo "RESAMPLE: {r}"

{PYTHON_BIN} {WORKER_PATH} \\
  --patient {patient} \\
  --resample-idx {r} \\
  --cluster-preds-path "{row['cluster_preds_path']}" \\
  --frs-path "{row['frs_path']}" \\
  --out-dir "{row['out_root']}" \\
  --params-json "{row['consensus_json']}" \\
  {word_idx_arg} \\
  --test-size {TEST_SIZE} \\
  --n-iter {SEARCH_ITERS} \\
  --cv-splits {CV_SPLITS} \\
  --n-shuffles {N_SHUFFLES} \\
  --seed-stride {SEED_STRIDE} \\
  --n-jobs {CPUS * 2} \\
  --scoring {SCORING} \\
  --perm-metric {PERM_METRIC}

echo "END: $(date)"
"""
    sbatch_path = scripts_dir / f'{patient}_{date_str}_resample_{r}.sbatch'
    sbatch_path.write_text(sbatch_text)

    if DRY_RUN:
        submitted.append((patient, date_str, r, 'dry-run'))
        print(f'dry-run: {patient} {date_str} r={r}')
    else:
        try:
            res = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
            submitted.append((patient, date_str, r, res.stdout.strip()))
            print(f'submitted: {patient} {date_str} r={r} -> {res.stdout.strip()}')
        except subprocess.CalledProcessError as exc:
            failed.append((patient, date_str, r, exc.stderr.strip()))
            print(f'FAILED: {patient} {date_str} r={r}\n{exc.stderr}')

print(f'phase-2 submitted={len(submitted)}, failed={len(failed)}')

submitted: YFC 2024-07-21 r=3 -> Submitted batch job 295720
submitted: YFC 2024-07-21 r=4 -> Submitted batch job 295721
submitted: YFC 2024-07-21 r=5 -> Submitted batch job 295722
submitted: YFC 2024-07-21 r=6 -> Submitted batch job 295723
submitted: YFC 2024-07-21 r=7 -> Submitted batch job 295724
submitted: YFC 2024-07-21 r=8 -> Submitted batch job 295725
submitted: YFC 2024-07-21 r=9 -> Submitted batch job 295726
submitted: YFC 2024-07-21 r=10 -> Submitted batch job 295727
submitted: YFC 2024-07-21 r=11 -> Submitted batch job 295728
submitted: YFC 2024-07-21 r=12 -> Submitted batch job 295729
submitted: YFC 2024-07-21 r=13 -> Submitted batch job 295730
submitted: YFC 2024-07-21 r=14 -> Submitted batch job 295731
submitted: YFC 2024-07-21 r=15 -> Submitted batch job 295732
submitted: YFC 2024-07-21 r=16 -> Submitted batch job 295733
submitted: YFC 2024-07-21 r=17 -> Submitted batch job 295734
submitted: YFC 2024-07-21 r=18 -> Submitted batch job 295735
submitted: YFC 2024-07-21 r=19 

### 8. Check Status

In [ ]:
status_df = build_status_df()
pair_progress = (
    status_df.groupby(['patient', 'date'])
    .agg(
        total=('resample_idx', 'count'),
        done=('has_success', 'sum'),
        errors=('has_error', 'sum'),
    )
    .reset_index()
)
pair_progress['pct'] = (pair_progress['done'] / pair_progress['total'] * 100).round(0).astype(int)
print(f'Overall: {int(status_df["has_success"].sum())} / {len(status_df)} resamples done')
display(pair_progress)

### 9. Inspect Errors

In [ ]:
error_rows = status_df[status_df['has_error']].copy()
print(f'resamples with error files: {len(error_rows)}')
for _, row in error_rows.head(10).iterrows():
    print('=' * 100)
    print(row['patient'], row['date'], f'r={row["resample_idx"]}')
    print(row['error_path'].read_text()[:4000])

### 10. Aggregate Results Per (Patient, Day)

In [ ]:
for (patient, date_str), grp in status_df.groupby(['patient', 'date']):
    if not bool(grp['has_success'].all()):
        print(f'skipping {patient} {date_str}: not all resamples finished')
        continue

    out_root     = grp.iloc[0]['out_root']
    summary_rows = []
    aggregate    = {}
    for r in range(N_RESAMPLES):
        with open(out_root / f'summary_resample_{r}.json') as f:
            summary_rows.append(json.load(f))
        with open(out_root / f'scat_sampled_resample_{r}_.pkl', 'rb') as f:
            aggregate[f'resample_{r}'] = pickle.load(f)

    summary_df = pd.DataFrame(summary_rows).sort_values('resample_idx').reset_index(drop=True)
    agg_csv = out_root / 'scat_sampled_all_resamples.csv'
    agg_pkl = out_root / 'scat_sampled_all_resamples.pkl'
    summary_df.to_csv(agg_csv, index=False)
    with open(agg_pkl, 'wb') as f:
        pickle.dump(aggregate, f)
    print(f'aggregated {patient} {date_str} -> {agg_csv}')
    display(summary_df)